In [ ]:
import os
import pandas as pd

BASE = "/kaggle/input/datasets/tanlikesmath/diabetic-retinopathy-resized"
IMG_DIR = f"{BASE}/resized_train_cropped/resized_train_cropped"

# Load labels and keep only the needed columns
df = pd.read_csv(f"{BASE}/trainLabels_cropped.csv")[["image", "level"]]
df["path"] = IMG_DIR + "/" + df["image"] + ".jpeg"
df = df[df["path"].apply(os.path.exists)]  # drop rows with missing images

print("Images per grade (full data):")
print(df["level"].value_counts().sort_index())

# Balanced sample: 50 images per grade (0-4), fixed seed for reproducibility
sample = df.groupby("level").sample(n=50, random_state=42).reset_index(drop=True)
sample.to_csv("/kaggle/working/sample_250.csv", index=False)

print("\nSample size:", len(sample))
print(sample["level"].value_counts().sort_index())

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Log in with the token stored in Kaggle secrets
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

# Load model across both T4 GPUs
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model loaded on:", model.device)

In [ ]:
import time
import pandas as pd
from PIL import Image

sample = pd.read_csv("/kaggle/working/sample_250.csv")
test_imgs = sample.groupby("level").head(1)  # one image per grade

PROMPT = ("Describe the findings in this retinal fundus image and "
          "assess the diabetic retinopathy severity.")

def generate_report(img_path, prompt, max_new_tokens=400):
    """Run MedGemma on one image and return the generated text."""
    image = Image.open(img_path).convert("RGB")
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are an expert ophthalmologist."}]},
        {"role": "user", "content": [{"type": "text", "text": prompt},
                                     {"type": "image", "image": image}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][input_len:], skip_special_tokens=True)



In [ ]:
STRUCTURED_PROMPT = """Analyze this retinal fundus image and respond ONLY in the exact format below.

Use the ICDR diabetic retinopathy scale:
0 = No DR: no abnormalities
1 = Mild NPDR: microaneurysms only
2 = Moderate NPDR: more than microaneurysms but less than severe
3 = Severe NPDR: extensive hemorrhages, venous beading, or IRMA, with no signs of proliferation
4 = Proliferative DR: neovascularization and/or vitreous/preretinal hemorrhage

IMAGE QUALITY: <Good / Adequate / Poor>
FINDINGS:
- Microaneurysms: <Present / Absent>
- Hemorrhages: <Present / Absent>
- Hard exudates: <Present / Absent>
- Cotton wool spots: <Present / Absent>
- Venous beading or IRMA: <Present / Absent>
- Neovascularization: <Present / Absent>
- Vitreous or preretinal hemorrhage: <Present / Absent>
SEVERITY GRADE: <a single number from 0 to 4>
SEVERITY LABEL: <No DR / Mild NPDR / Moderate NPDR / Severe NPDR / Proliferative DR>
RECOMMENDATION: <one actionable follow-up sentence>
SUMMARY: <2-3 simple sentences for a spoken audio summary>"""



In [ ]:
import os, time
import pandas as pd

OUT = "/kaggle/working/reports.csv"
sample = pd.read_csv("/kaggle/working/sample_250.csv")

# Structured first so the summaries are ready early for the audio part
PROMPTS = {"structured": (STRUCTURED_PROMPT, 350), "zero_shot": (PROMPT, 512)}

# Resume support: skip (image, prompt) pairs already saved
done = set()
if os.path.exists(OUT):
    prev = pd.read_csv(OUT)
    done = set(zip(prev["image"], prev["prompt_type"]))

def flush(rows):
    """Append buffered rows to the CSV file."""
    pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)

buffer = []
for name, (prompt, max_tok) in PROMPTS.items():
    for i, row in sample.iterrows():
        if (row["image"], name) in done:
            continue
        t0 = time.time()
        report = generate_report(row["path"], prompt, max_new_tokens=max_tok)
        buffer.append({"image": row["image"], "true_grade": row["level"],
                       "prompt_type": name, "report": report,
                       "seconds": round(time.time() - t0, 1)})
        if len(buffer) >= 10:  # save every 10 reports
            flush(buffer); buffer = []
            print(f"{name}: {i + 1}/{len(sample)}")
if buffer:
    flush(buffer)
print("All done!")

In [ ]:
results = pd.read_csv(OUT)

# Number of reports and average time per prompt type
print(results.groupby("prompt_type").agg(reports=("image", "count"),
                                         avg_seconds=("seconds", "mean")))